<a href="https://colab.research.google.com/github/akatsushikami-arch/HW/blob/main/%D0%9F%D0%BE%D1%80%D0%BE%D0%B4%D1%8B_%D1%81%D0%BE%D0%B1%D0%B0%D0%BA_%D1%81%D0%BB%D0%BE%D0%B6%D0%BD%D0%B0%D1%8F_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D1%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

In [2]:
!pip install importlib_resources

In [3]:
dataset, info = tfds.load('oxford_iiit_pet', with_info=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.P69J1Y_4.0.0/oxford_iiit_pet-train.tfrecord-[0-…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.P69J1Y_4.0.0/oxford_iiit_pet-test.tfrecord-[0-9…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.


In [4]:
def is_dog(sample):
    return sample['species'] == 1

In [5]:
train_dogs = dataset['train'].filter(is_dog)
test_dogs = dataset['test'].filter(is_dog)

In [6]:
dog_labels = [i for i, name in enumerate(info.features['label'].names) if name.islower()]
NUM_CLASSES = len(dog_labels)

In [7]:
lookup_table = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(dog_labels, dtype=tf.int64),
        values=tf.constant(list(range(NUM_CLASSES)), dtype=tf.int64)
    ),
    default_value=-1
)

In [8]:
IMG_SIZE = 224
BATCH_SIZE = 32

In [9]:
def preprocess(sample):
    image = tf.image.resize(sample['image'], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    label = lookup_table.lookup(sample['label'])
    return image, label

In [10]:
train_ds = train_dogs.map(preprocess).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_dogs.map(preprocess).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [11]:
base_model = ResNet50(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [12]:
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_layer')

x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.08)(x)
x = layers.RandomZoom(0.1)(x)

x = preprocess_input(x)
x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer')(x)

resnet_model = models.Model(inputs=inputs, outputs=outputs, name='resnet50_dogs')
resnet_model.summary()

Model: "resnet50_dogs"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip         │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (RandomFlip)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation     │ (None, 224, 224,  │          0 │ random_flip[0][0] │
│ (RandomRotation)    │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_zoom         │ (None, 224, 224,  │          0 │ random_rotation[… │
│ (RandomZoom)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ random_zoom[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ random_zoom[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ random_zoom[0][0] │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer        │ (None, 25)        │      6,425 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,118,681 (92.01 MB)

 Trainable params: 530,969 (2.03 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [13]:
lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

early_stopping = callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=6,
    restore_best_weights=True,
    verbose=1
)

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = resnet_model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    callbacks=[lr_scheduler, early_stopping]
)

Epoch 1/15
     78/Unknown 30s 165ms/step - accuracy: 0.3284 - loss: 2.5511

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


78/78 ━━━━━━━━━━━━━━━━━━━━ 45s 369ms/step - accuracy: 0.5048 - loss: 1.7881 - val_accuracy: 0.8592 - val_loss: 0.4927 - learning_rate: 0.0010
Epoch 2/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 20s 256ms/step - accuracy: 0.7636 - loss: 0.7749 - val_accuracy: 0.8793 - val_loss: 0.4058 - learning_rate: 0.0010
Epoch 3/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 20s 258ms/step - accuracy: 0.8154 - loss: 0.6200 - val_accuracy: 0.8914 - val_loss: 0.3683 - learning_rate: 0.0010
Epoch 4/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 20s 259ms/step - accuracy: 0.8387 - loss: 0.5157 - val_accuracy: 0.8946 - val_loss: 0.3276 - learning_rate: 0.0010
Epoch 5/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 20s 263ms/step - accuracy: 0.8579 - loss: 0.4559 - val_accuracy: 0.9055 - val_loss: 0.3026 - learning_rate: 0.0010
Epoch 6/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 20s 262ms/step - accuracy: 0.8864 - loss: 0.3599 - val_accuracy: 0.9039 - val_loss: 0.3240 - learning_rate: 0.0010
Epoch 7/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 21s 267ms/step - accuracy: 0.8776 - loss: 0.3813 - val_